In [1]:
import requests
import tiktoken
import torch
from torch import nn
import datasets


tokenizer = tiktoken.get_encoding("gpt2")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [2]:
engine_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/model.py")

with open("model.py", "w") as f:
    f.write(engine_res.text)
    # print(engine_res.text)
print("Downloaded model.py")

# Downloading train_utils.py
train_utils_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/train_utils.py")

with open("train_utils.py", "w") as f:
    f.write(train_utils_res.text)
    # print(engine_res.text)
print("Downloaded train_utils.py")

data_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/data.py")

with open("data.py", "w") as f:
    f.write(data_res.text)
    # print(engine_res.text)
print("Downloaded data.py")

#Downloading custom_gpt_v2.pth weights
wt_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/Models/tara_n1_pretrain_v2.pth")
with open("tara_n1_pretrain_v2.pth", "wb") as f:
    f.write(wt_res.content)
print("Downloaded modle weights")


Downloaded model.py
Downloaded train_utils.py
Downloaded data.py
Downloaded modle weights


In [3]:
from model import *
from train_utils import *
from data import *


config = GPTConfig(
    vocab_size=tokenizer.n_vocab,
    block_size=1024,
    batch_size=4,
    d_model=256,
    hidden_layers=1024,
    n_heads=4,
    n_layers=6,
)

# Mid Trainign with `SimpleStories/SimpleStories`, `pszemraj/simple_wikipedia_LM`, and  `rojagtap/bookcorpus` datasets

## Download and prep Dataset


In [4]:
from datasets import load_dataset

simple_stories_ds = load_dataset("SimpleStories/SimpleStories")
simple_wiki_ds = load_dataset("pszemraj/simple_wikipedia_LM")
bookcorpus_ds = load_dataset("rojagtap/bookcorpus")


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00007.parquet:   0%|          | 0.00/238M [00:00<?, ?B/s]

data/train-00001-of-00007.parquet:   0%|          | 0.00/238M [00:00<?, ?B/s]

data/train-00002-of-00007.parquet:   0%|          | 0.00/238M [00:00<?, ?B/s]

data/train-00003-of-00007.parquet:   0%|          | 0.00/238M [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/5943 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

books_large_p1.txt:   0%|          | 0.00/2.52G [00:00<?, ?B/s]

books_large_p2.txt:   0%|          | 0.00/2.10G [00:00<?, ?B/s]

Generating train split:   0%|          | 0/74004228 [00:00<?, ? examples/s]

In [5]:
print(len(simple_stories_ds))
print(len(simple_wiki_ds))
print(len(bookcorpus_ds))

2
3
1


In [6]:
bookcorpus_ds["train"][0:10]

{'text': ['the half-ling book one in the fall of igneeria series kaylee soderburg copyright 2013 kaylee soderburg all rights reserved .',
  'isbn : 1492913731 isbn-13 : 978-1492913733 for my family , who encouraged me to never stop fighting for my dreams chapter 1 summer vacations supposed to be fun , right ?',
  'i wish i had a better answer to that question .',
  'starlings , new york is not the place youd expect much to happen .',
  'its a small quiet town , the kind where everyone knows your name .',
  'its a place where your parents wouldnt even care if you stayed out late biking with your friends .',
  'only because everyone felt so safe , so comfy .',
  'they dont know the half of it .',
  'but i do .',
  'i know it all and starlings is not the place where you want to be after dark .']}

In [7]:
import numpy as np
import random

simple_stories_examples = []
simple_wiki_examples = []
bookcorpus_examples = []

for sample in simple_stories_ds["train"]:
    simple_stories_examples.append(sample["story"])
    if len(simple_stories_examples) == 280000:
        break
print("Collected Simple Stories Examples")

for sample in simple_wiki_ds["train"]:
    simple_wiki_examples.append("# Title: "+sample["title"]+"\n"+sample["text"])
    if len(simple_wiki_examples) == 320000:
        break
print("Collected Simple Wiki Examples")

for i in range(0, len(bookcorpus_ds["train"]), 10):

    sample = [x.replace(" !", "!").replace(" ,", ",").replace(" .", ".") for x in bookcorpus_ds["train"][i:i+10]["text"]]

    bookcorpus_examples.append(" ".join(sample))

    if len(bookcorpus_examples) == 200000:
        break
print("Collected BookCoupus Examples")


print("Creating interleaved data")

choices = (["a"]*len(simple_stories_examples)) + (["b"]*len(simple_wiki_examples)) + (["c"]*len(bookcorpus_examples))

random.shuffle(choices)

data = []

for choice in choices:
    if choice == "a":
        data.append(simple_stories_examples.pop(0))
    elif choice == "b":
        data.append(simple_wiki_examples.pop(0))
    else:
        data.append(bookcorpus_examples.pop(0))

print("Interleaved dataset created")





Collected Simple Stories Examples
Collected Simple Wiki Examples
Collected BookCoupus Examples
Creating interleaved data
Interleaved dataset created


In [8]:
from tqdm.auto import tqdm
idx = 0
eot = tokenizer.eot_token
# pbar = tqdm(total = len(data), )
total_tokens = 0
for text in tqdm(data, desc="Calculating token counts"):
    # Tokenize + 1 for the <|endoftext|> delimiter
    tokens = tokenizer.encode(text, allowed_special="all")
    total_tokens += len(tokens) + 2

print(f"Total tokens: {total_tokens:,}")

mmap = np.lib.format.open_memmap("mid_training.npy", dtype=np.uint16, mode="w+", shape=(total_tokens, ))
for i in tqdm(range(1, len(data)+1), desc="Creating tokens"):
    text = data[i-1]+"\n"
    tokens = tokenizer.encode(text, allowed_special="all")
    tokens.append(eot)

    length = len(tokens)
    mmap[idx: idx+length] = np.array(tokens, dtype=np.uint16)
    idx += length

mmap.flush()
print("Successfullly saved tokens.")

Calculating token counts:   0%|          | 0/705984 [00:00<?, ?it/s]

Total tokens: 165,987,170


Creating tokens:   0%|          | 0/705984 [00:00<?, ?it/s]

Successfullly saved tokens.


### Creating dataloaders

In [9]:
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import numpy as np
class MidTrainDataset(Dataset):
    def __init__(self, path, block_size):
        self.data = np.load(path, mmap_mode="r", allow_pickle=True)
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx:idx+self.block_size]
        y = self.data[idx+1:idx+self.block_size+1]

        return (
            torch.from_numpy(x.astype(np.int64)),
            torch.from_numpy(y.astype(np.int64))
        )


def create_data_doc(data):
    doc = ""
    pbar = tqdm(total=len(data))
    for sample in data:
        text = sample["story"]
        doc += f"{text}\n{tokenizer.decode([tokenizer.eot_token])}\n\n"
        pbar.update(1)


    return doc.strip()

In [10]:
full_dataset = MidTrainDataset("mid_training.npy", config.block_size)

train_split = int(0.9*len(full_dataset))

train_dataset = torch.utils.data.Subset(full_dataset, range(0, train_split))
test_dataset = torch.utils.data.Subset(full_dataset, range(train_split, len(full_dataset)))

train_dataloader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, pin_memory=True)
test_dataloader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=True, pin_memory=True)

print(len(train_dataset))
print(len(test_dataset))

149387531
16598615


In [11]:
print(len(test_dataset))

16598615


## Training loop

In [12]:
model = CustomGPT(config)
state_dict = torch.load("tara_n1_pretrain_v2.pth", map_location=device)
state_dict = {k.removeprefix("module."):v for k, v in state_dict.items()}
model.load_state_dict(state_dict)

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    print(f"Using {torch.cuda.device_count()} GPUs")

model.to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler()

Using 2 GPUs


In [13]:
from tqdm.auto import tqdm

steps = 20822
test_count = 5000
test_steps = 1000

train_iter = iter(train_dataloader)

avg_train_loss = 0

for step in tqdm(range(1, steps+1)):
    model.train()
    if not train_iter:
        train_iter = iter(train_dataloader)

    train_loss = train_step(model, train_iter, loss_fn, optimizer, scaler, device, accumulation_steps=1)
    avg_train_loss += train_loss
    if step%test_count == 0:
        avg_train_loss /= test_count

        test_loss = test_step(model, test_dataloader, test_steps, loss_fn, device)

        print(f"Step: {step} | Train loss: {avg_train_loss} | Test loss: {test_loss}")
        avg_train_loss = 0


  0%|          | 0/20822 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

Step: 5000 | Train loss: 3.272632662463188 | Test loss: 3.171803691625595


  0%|          | 0/1000 [00:00<?, ?it/s]

Step: 10000 | Train loss: 3.1001180898427965 | Test loss: 3.0962177043557166


  0%|          | 0/1000 [00:00<?, ?it/s]

Step: 15000 | Train loss: 3.0177437023043634 | Test loss: 3.0741238256692887


  0%|          | 0/1000 [00:00<?, ?it/s]

Step: 20000 | Train loss: 2.9724212849378584 | Test loss: 3.0509001163244247


In [14]:
torch.save(model.state_dict(), "tara_n1_pretrain_v3.pth")

In [15]:
test_model = CustomGPT(config)
state_dict = torch.load("tara_n1_pretrain_v3.pth", map_location=device)
state_dict = {k.removeprefix("module."):v for k, v in state_dict.items()}
test_model.load_state_dict(state_dict)

test_model.to(device)

CustomGPT(
  (token_embeddings): Embedding(50257, 256)
  (positional_embeddings): Embedding(1024, 256)
  (transformer_blocks): ModuleList(
    (0-5): 6 x TransformerBlock(
      (multi_head_attention): MultiHeadAttention(
        (query): Linear(in_features=256, out_features=256, bias=True)
        (key): Linear(in_features=256, out_features=256, bias=True)
        (value): Linear(in_features=256, out_features=256, bias=True)
        (out_proj): Linear(in_features=256, out_features=256, bias=True)
      )
      (mlp): MLP(
        (mpl_layer): Sequential(
          (0): Linear(in_features=256, out_features=1024, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=1024, out_features=256, bias=True)
        )
      )
      (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    )
  )
  (layer_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (lm_head): Linear(in_featu

In [16]:
EVAL_PROMPTS = [
    "The history of computers began",
    "Artificial intelligence has changed the way",
    "The scientist entered the laboratory and",
    "In computer science, a binary tree is",
    "The development of the internet",
    "The human brain is capable of",
    "Climate change is one of the major challenges",
    "The process of photosynthesis begins when",
    "A software engineer working on a large project",
    "The discovery of electricity",
    "The Earth's atmosphere consists mainly of",
    "In mathematics, prime numbers are",
    "The university library was unusually quiet because",
    "After several hours of testing, the researchers",
    "The small village was located near",
    "When the spacecraft entered the atmosphere",
    "Learning a new programming language requires",
    "The ancient city was known for",
    "The doctor examined the patient and",
    "One of the most important problems in modern society",
]

In [17]:
#256 Tokens
import json
generations = {}
for prompt in EVAL_PROMPTS:
    context = torch.tensor(tokenizer.encode(prompt), dtype=torch.long, device=device).unsqueeze(0)

    with torch.inference_mode():
        generated = test_model.generate(context, max_new_tokens=256)

    result = tokenizer.decode(generated[0].tolist())
    generations[prompt] = result

with open("eval_ganaration_256.json", "w") as f:
    json.dump(generations, f)

print("256 Token Generations stored in eval_ganaration_256.json")

256 Token Generations stored in eval_ganaration_256.json


In [18]:
#512 Tokens
import json
generations = {}

for prompt in EVAL_PROMPTS:
    context = torch.tensor(tokenizer.encode(prompt), dtype=torch.long, device=device).unsqueeze(0)

    with torch.inference_mode():
        generated = test_model.generate(context, max_new_tokens=512)

    result = tokenizer.decode(generated[0].tolist())
    generations[prompt] = result

with open("eval_ganaration_512.json", "w") as f:
    json.dump(generations, f)

print("512 Token Generations stored in eval_ganaration_512.json")


512 Token Generations stored in eval_ganaration_512.json


In [19]:
#1024 Tokens
import json
generations = {}

for prompt in EVAL_PROMPTS:
    context = torch.tensor(tokenizer.encode(prompt), dtype=torch.long, device=device).unsqueeze(0)

    with torch.inference_mode():
        generated = test_model.generate(context, max_new_tokens=1024)

    result = tokenizer.decode(generated[0].tolist())
    generations[prompt] = result

with open("eval_ganaration_1024.json", "w") as f:
    json.dump(generations, f)

print("1024 Token Generations stored in eval_ganaration_1024.json")


1024 Token Generations stored in eval_ganaration_1024.json
